# **HW15 Meta Learning: Few-shot Classification**

Useful Links:
1. [Go to hyperparameter setting.](#hyp)
1. [Go to meta algorithm setting.](#modelsetting)
1. [Go to main loop.](#mainloop)

## **Step 0: Check GPU**

In [ ]:
!nvidia-smi

In [ ]:
# 安裝 qqdm（進度條套件）
try:
    import qqdm
except:
    !pip install qqdm > /dev/null 2>&1
print('Done!')

## **Step 1: Download Data**

執行下方 cell 下載資料集（已由 TA 預處理完成，包含資料增強，不需再自行增強）。

**版本注意**：gdown >= 4.x 移除了 `--id` 參數，改用直接傳入 Google Drive URL 的方式。

In [ ]:
workspace_dir = '.'

# 版本修正：gdown 4.x 以後棄用 --id，改用 URL 格式
!gdown 'https://drive.google.com/uc?id=1FLDrQ0k-iJ-mk8ors0WItqvwgu0w9J0U' \
    --output "{workspace_dir}/Omniglot.tar.gz"

In [ ]:
# 解壓縮資料集
!tar -zxf "{workspace_dir}/Omniglot.tar.gz" \
    -C "{workspace_dir}/"

### Data Preview

In [ ]:
from PIL import Image
from IPython.display import display

# 預覽日文平假名字元影像
for i in range(10, 20):
    im = Image.open(
        'Omniglot/images_background/Japanese_(hiragana).0/character13/0500_'
        + str(i) + '.png')
    display(im)

## **Step 2: Build the model**

### Library importation

In [ ]:
import glob, random
from collections import OrderedDict

import numpy as np

try:
    from qqdm.notebook import qqdm as tqdm
except ModuleNotFoundError:
    from tqdm.auto import tqdm

import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms

from PIL import Image
from IPython.display import display

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 固定隨機種子確保可重現性
random_seed = 0
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

### Model Construction Preliminaries

任務是影像分類，因此使用 CNN。
MAML 的特殊之處：Outer loop 要對原始參數 θ 求梯度，
而非 Inner loop 更新後的 θ'，因此需要 `functional_forward`
（手動傳入參數）而非直接用 `nn.Module.forward`。

### Model block definition

In [ ]:
def ConvBlock(in_ch: int, out_ch: int):
    """標準 Conv -> BN -> ReLU -> MaxPool 積木"""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

def ConvBlockFunction(x, w, b, w_bn, b_bn):
    """Functional 版本：手動傳入參數，讓梯度可以追蹤到原始 θ"""
    x = F.conv2d(x, w, b, padding=1)
    # running_mean/var=None + training=True 表示使用 batch 統計量
    x = F.batch_norm(x,
                     running_mean=None,
                     running_var=None,
                     weight=w_bn, bias=b_bn,
                     training=True)
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=2, stride=2)
    return x

### Model definition

In [ ]:
class Classifier(nn.Module):
    def __init__(self, in_ch, k_way):
        super(Classifier, self).__init__()
        self.conv1 = ConvBlock(in_ch, 64)
        self.conv2 = ConvBlock(64, 64)
        self.conv3 = ConvBlock(64, 64)
        self.conv4 = ConvBlock(64, 64)
        self.logits = nn.Linear(64, k_way)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = x.view(x.shape[0], -1)
        x = self.logits(x)
        return x

    def functional_forward(self, x, params):
        '''
        Arguments:
        x: input images [batch, 1, 28, 28]
        params: OrderedDict，包含各層 weight/bias（含 BN 參數）

        手動把參數帶入每層計算，讓 autograd 可以追蹤到原始 θ 的梯度
        '''
        for block in [1, 2, 3, 4]:
            x = ConvBlockFunction(
                x,
                params[f'conv{block}.0.weight'],
                params[f'conv{block}.0.bias'],
                params.get(f'conv{block}.1.weight'),
                params.get(f'conv{block}.1.bias'))
        x = x.view(x.shape[0], -1)
        x = F.linear(x,
                     params['logits.weight'],
                     params['logits.bias'])
        return x

### Create Label

N-way K-shot 設定下，每個 task 有 n_way 個類別，
每個類別有 k_shot 張圖，建立對應的 label tensor。

In [ ]:
def create_label(n_way, k_shot):
    return (
        torch.arange(n_way)
             .repeat_interleave(k_shot)
             .long()
    )

# 驗證：5-way 2-shot 的 label
create_label(5, 2)

### Accuracy calculation

In [ ]:
def calculate_accuracy(logits, val_label):
    """計算預測準確率"""
    acc = np.asarray([(
        torch.argmax(logits, -1).cpu().numpy() == val_label.cpu().numpy())]
        ).mean()
    return acc

### Define Dataset

每個 `__getitem__` 回傳一個字元的 (k_shot + q_query) 張圖，
大小為 `[k_shot+q_query, 1, 28, 28]`。

In [ ]:
class Omniglot(Dataset):
    def __init__(self, data_dir, k_way, q_query):
        self.file_list = [f for f in glob.glob(
            data_dir + '**/character*',
            recursive=True)]
        self.transform = transforms.Compose(
                            [transforms.ToTensor()])
        self.n = k_way + q_query

    def __getitem__(self, idx):
        sample = np.arange(20)
        np.random.shuffle(sample)   # 隨機抽樣
        img_path = self.file_list[idx]
        img_list = [f for f in glob.glob(
            img_path + '**/*.png', recursive=True)]
        img_list.sort()
        imgs = [self.transform(
            Image.open(img_file))
            for img_file in img_list]
        imgs = torch.stack(imgs)[sample[:self.n]]
        return imgs

    def __len__(self):
        return len(self.file_list)

## **Step 3: Core MAML**

以下是完整的 MAML 算法（參考論文實作，inner_train_step=1 時為二階 MAML）。
- **Support set**：inner loop 的訓練資料
- **Query set**：outer loop 的驗證資料（用於更新原始參數 θ）

In [ ]:
def OriginalMAML(
    model, optimizer, x, n_way, k_shot, q_query, loss_fn,
    inner_train_step=1, inner_lr=0.4, train=True):
    criterion, task_loss, task_acc = loss_fn, [], []

    for meta_batch in x:
        # 切分 support / query set
        support_set = meta_batch[: n_way * k_shot]
        query_set   = meta_batch[n_way * k_shot :]

        # 複製參數供 inner loop 使用
        fast_weights = OrderedDict(model.named_parameters())

        ### ---------- INNER TRAIN LOOP ---------- ###
        for inner_step in range(inner_train_step):
            train_label = create_label(n_way, k_shot).to(device)
            logits = model.functional_forward(support_set, fast_weights)
            loss   = criterion(logits, train_label)

            # create_graph=True：保留計算圖以計算二階梯度
            grads = torch.autograd.grad(
                loss, fast_weights.values(),
                create_graph=True)
            # SGD 更新 fast_weights（不更新模型本身的參數）
            fast_weights = OrderedDict(
                (name, param - inner_lr * grad)
                for ((name, param), grad)
                    in zip(fast_weights.items(), grads))

        ### ---------- INNER VALID LOOP ---------- ###
        val_label = create_label(n_way, q_query).to(device)
        logits = model.functional_forward(query_set, fast_weights)
        loss   = criterion(logits, val_label)
        task_loss.append(loss)
        task_acc.append(calculate_accuracy(logits, val_label))

    # Outer loop 更新原始模型參數
    model.train()
    optimizer.zero_grad()
    meta_batch_loss = torch.stack(task_loss).mean()
    if train:
        meta_batch_loss.backward()
        optimizer.step()
    task_acc = np.mean(task_acc)
    return meta_batch_loss, task_acc

## Variations of MAML

### First-order approximation of MAML (FOMAML)
略去二階梯度（Hessian）計算，用 `create_graph=False` 降低計算量。

### Almost No Inner Loop (ANIL)
來自 [Raghu et al. 2020](https://arxiv.org/abs/1909.09157)，
Inner loop **只更新分類頭**（最後一層），Feature extractor 不動。
利用「Feature Reuse」特性減少計算量。

以下將 MAML 中三個可替換的部分抽成函式，
選擇不同函式組合即可得到 FOMAML / ANIL。

### Part 1: Inner loop update

MAML（原版）

In [ ]:
def inner_update_MAML(fast_weights, loss, inner_lr):
    """二階 MAML：create_graph=True 保留計算圖"""
    grads = torch.autograd.grad(
        loss, fast_weights.values(), create_graph=True)
    fast_weights = OrderedDict(
        (name, param - inner_lr * grad)
        for ((name, param), grad) in zip(fast_weights.items(), grads))
    return fast_weights

Alternatives

In [ ]:
def inner_update_alt1(fast_weights, loss, inner_lr):
    """
    FOMAML：create_graph=False，不保留二階梯度計算圖。
    數學上等價於只用一階梯度近似（First-Order Approximation）。
    """
    grads = torch.autograd.grad(
        loss, fast_weights.values(), create_graph=False)
    fast_weights = OrderedDict(
        (name, param - inner_lr * grad)
        for ((name, param), grad) in zip(fast_weights.items(), grads))
    return fast_weights

def inner_update_alt2(fast_weights, loss, inner_lr):
    """
    ANIL：只對最後兩個參數（logits.weight, logits.bias）求梯度並更新。
    Feature extractor 在 inner loop 完全不動（Almost No Inner Loop）。
    """
    grads = torch.autograd.grad(
        loss, list(fast_weights.values())[-2:], create_graph=True)
    for ((name, param), grad) in zip(
            list(fast_weights.items())[-2:], grads):
        fast_weights[name] = param - inner_lr * grad
    return fast_weights

### Part 2: Collect gradients

MAML（不做特殊處理，由 PyTorch autograd 自動計算）

In [ ]:
def collect_gradients_MAML(
    special_grad: OrderedDict, fast_weights, model, len_data):
    """MAML 不需手動收集梯度，直接由 backward() 處理"""
    return special_grad

Alternatives

In [ ]:
def collect_gradients_alt(
    special_grad: OrderedDict, fast_weights, model, len_data):
    """
    手動計算梯度：用初始參數與 fast_weights 的差值作為梯度。
    （Reptile 風格的 outer loop 更新）
    """
    diff = OrderedDict(
        (name, params - fast_weights[name])
        for (name, params) in model.named_parameters())
    for name in diff:
        special_grad[name] = (
            special_grad.get(name, 0) + diff[name] / len_data)
    return special_grad

### Part 3: Outer loop gradients calculation

MAML（直接呼叫 PyTorch backward）

In [ ]:
def outer_update_MAML(model, meta_batch_loss, grad_tensors):
    """MAML/FOMAML/ANIL 的 outer update：直接 backward"""
    meta_batch_loss.backward()

Alternatives

In [ ]:
def outer_update_alt(model, meta_batch_loss, grad_tensors):
    """將預先計算好的梯度張量直接塞入參數的 .grad 欄位"""
    for (name, params) in model.named_parameters():
        params.grad = grad_tensors[name]

### Complete the algorithm

用 `MetaAlgorithmGenerator` 組合三個積木，得到目標演算法。
預設三個積木都填入 MAML 版本。

In [ ]:
def MetaAlgorithmGenerator(
    inner_update      = inner_update_MAML,
    collect_gradients = collect_gradients_MAML,
    outer_update      = outer_update_MAML):

    global calculate_accuracy

    def MetaAlgorithm(
        model, optimizer, x, n_way, k_shot, q_query, loss_fn,
        inner_train_step=1, inner_lr=0.4, train=True):
        criterion = loss_fn
        task_loss, task_acc = [], []
        special_grad = OrderedDict()  # 供 collect_gradients_alt 使用

        for meta_batch in x:
            support_set = meta_batch[: n_way * k_shot]
            query_set   = meta_batch[n_way * k_shot :]

            fast_weights = OrderedDict(model.named_parameters())

            ### ---------- INNER TRAIN LOOP ---------- ###
            for inner_step in range(inner_train_step):
                train_label = create_label(n_way, k_shot).to(device)
                logits = model.functional_forward(
                    support_set, fast_weights)
                loss   = criterion(logits, train_label)
                fast_weights = inner_update(
                    fast_weights, loss, inner_lr)

            ### ---------- INNER VALID LOOP ---------- ###
            val_label = create_label(n_way, q_query).to(device)
            special_grad = collect_gradients(
                special_grad, fast_weights, model, len(x))

            logits = model.functional_forward(
                query_set, fast_weights)
            loss   = criterion(logits, val_label)
            task_loss.append(loss)
            task_acc.append(
                calculate_accuracy(logits, val_label))

        # Outer loop 更新
        model.train()
        optimizer.zero_grad()
        meta_batch_loss = torch.stack(task_loss).mean()
        if train:
            outer_update(model, meta_batch_loss, special_grad)
            optimizer.step()
        task_acc = np.mean(task_acc)
        return meta_batch_loss, task_acc

    return MetaAlgorithm

### 三種演算法定義

| 演算法 | inner_update | 說明 |
|--------|-------------|------|
| MAML   | `inner_update_MAML`  | 二階梯度，全層更新 |
| FOMAML | `inner_update_alt1`  | 一階近似（`create_graph=False`），全層更新 |
| ANIL   | `inner_update_alt2`  | 只更新分類頭，feature extractor 不動 |

In [ ]:
# MAML：原始版本，二階梯度，所有層都在 inner loop 更新
MAML   = MetaAlgorithmGenerator()

# FOMAML：一階近似，create_graph=False，省去 Hessian 計算
FOMAML = MetaAlgorithmGenerator(
    inner_update=inner_update_alt1)

# ANIL：inner loop 只更新最後兩個參數（分類頭），Feature extractor 凍結
ANIL   = MetaAlgorithmGenerator(
    inner_update=inner_update_alt2)

## **Step 4: Initialization**

<a name="hyp"></a>
### Hyperparameters

In [ ]:
n_way            = 5      # 每個 task 有幾個類別
k_shot           = 1      # 每個類別的 support 樣本數
q_query          = 1      # 每個類別的 query 樣本數
inner_train_step = 1      # inner loop 更新次數
inner_lr         = 0.4    # inner loop learning rate
meta_lr          = 0.001  # outer loop learning rate
meta_batch_size  = 32     # 每次 meta-gradient 更新用的 task 數
max_epoch        = 30     # 總訓練 epoch 數
eval_batches     = 20     # 驗證時使用的 meta-batch 數
test_batches     = 20     # 測試時使用的 meta-batch 數
train_data_path  = './Omniglot/images_background/'
test_data_path   = './Omniglot/images_evaluation/'

### Dataloader initialization

In [ ]:
def dataloader_init(datasets, num_workers=2):
    train_set, val_set, test_set = datasets
    train_loader = DataLoader(
        train_set,
        batch_size=n_way,     # 每個 batch 取 n_way 個不同字元
        num_workers=num_workers,
        shuffle=True,
        drop_last=True)
    val_loader = DataLoader(
        val_set,
        batch_size=n_way,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True)
    test_loader = DataLoader(
        test_set,
        batch_size=n_way,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True)
    train_iter = iter(train_loader)
    val_iter   = iter(val_loader)
    test_iter  = iter(test_loader)
    return (train_loader, val_loader, test_loader), \
           (train_iter,  val_iter,   test_iter)

full_train_set = Omniglot(train_data_path, k_shot, q_query)

# 版本修正：動態計算 split 大小，避免寫死數字導致資料集大小不符時 crash
total = len(full_train_set)
val_size   = total // 6          # 約 1/6 作為驗證集
train_size = total - val_size
train_set, val_set = torch.utils.data.random_split(
    full_train_set, [train_size, val_size])
test_set = Omniglot(test_data_path, k_shot, q_query)

(train_loader, val_loader, test_loader), \
(train_iter,  val_iter,  test_iter) = dataloader_init(
    (train_set, val_set, test_set))

### Model & optimizer initialization

In [ ]:
def model_init():
    meta_model = Classifier(1, n_way).to(device)
    optimizer  = torch.optim.Adam(
        meta_model.parameters(), lr=meta_lr)
    loss_fn    = nn.CrossEntropyLoss().to(device)
    return meta_model, optimizer, loss_fn

meta_model, optimizer, loss_fn = model_init()

### Utility function to get a meta-batch

In [ ]:
def get_meta_batch(
        meta_batch_size, k_shot, q_query,
        data_loader, iterator):
    data = []
    for _ in range(meta_batch_size):
        try:
            # 版本修正：Python 3 的 iterator 使用 next()，
            # 原始碼的 iterator.next() 在 Python 3.10+ 會 AttributeError
            task_data = next(iterator)
        except StopIteration:
            iterator  = iter(data_loader)
            task_data = next(iterator)
        train_data = (
            task_data[:, :k_shot]
            .reshape(-1, 1, 28, 28))
        val_data   = (
            task_data[:, k_shot:]
            .reshape(-1, 1, 28, 28))
        task_data  = torch.cat((train_data, val_data), 0)
        data.append(task_data)
    return torch.stack(data).to(device), iterator

<a name="modelsetting"></a>
### Choose the meta learning algorithm

In [ ]:
# 可改為 FOMAML 或 ANIL
MetaAlgorithm = MAML

## **Step 5: Main program for training & testing**

<a name="mainloop"></a>
### Start training!

In [ ]:
for epoch in range(max_epoch):
    print('Epoch %d' % (epoch + 1))
    train_meta_loss = []
    train_acc       = []

    for step in tqdm(
            range(len(train_loader) // meta_batch_size)):
        x, train_iter = get_meta_batch(
            meta_batch_size, k_shot, q_query,
            train_loader, train_iter)
        meta_loss, acc = MetaAlgorithm(
            meta_model, optimizer, x,
            n_way, k_shot, q_query, loss_fn)
        train_meta_loss.append(meta_loss.item())
        train_acc.append(acc)

    print('  Loss    : ', '%.3f' % np.mean(train_meta_loss), end='\t')
    print('  Accuracy: ', '%.3f %%' % (np.mean(train_acc) * 100))

    # 每個 epoch 結束後進行驗證（inner loop 用 3 步）
    val_acc = []
    for eval_step in tqdm(
            range(len(val_loader) // eval_batches)):
        x, val_iter = get_meta_batch(
            eval_batches, k_shot, q_query,
            val_loader, val_iter)
        _, acc = MetaAlgorithm(
            meta_model, optimizer, x,
            n_way, k_shot, q_query, loss_fn,
            inner_train_step=3,
            train=False)
        val_acc.append(acc)
    print('  Validation accuracy: ',
          '%.3f %%' % (np.mean(val_acc) * 100))

### Testing the result

In [ ]:
test_acc = []
for test_step in tqdm(
        range(len(test_loader) // test_batches)):
    x, test_iter = get_meta_batch(
        test_batches, k_shot, q_query,
        test_loader, test_iter)
    # 測試時 inner loop 同樣用 3 步
    _, acc = MetaAlgorithm(
        meta_model, optimizer, x,
        n_way, k_shot, q_query, loss_fn,
        inner_train_step=3, train=False)
    test_acc.append(acc)
print('  Testing accuracy: ',
      '%.3f %%' % (np.mean(test_acc) * 100))

## **Reference**
1. Chelsea Finn, Pieter Abbeel, & Sergey Levine. (2017). [Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks.](https://arxiv.org/abs/1703.03400)
2. Aniruddh Raghu, Maithra Raghu, Samy Bengio, & Oriol Vinyals. (2020). [Rapid Learning or Feature Reuse? Towards Understanding the Effectiveness of MAML.](https://arxiv.org/abs/1909.09157)